**Now, we will stratify the data on the basis of different indicator values, including volatility, trading volume, and market sector**

These distinct strata will allow our models to better generalize trends between similar data and avoid any confusion

First, we make necessary imports

In [1]:
import glob
import os
import json
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
files = glob.glob('data_raw/*.parquet')
stats = []

for f in files:
    df = pd.read_parquet(f)
    
    stats.append({
        'path': f,
        'vol': df['VOL_20'].median(),
        'dollar_vol': (df['Close'] * df['Volume']).mean(),
        'sector': df['Market_Category'].iloc[0]
    })
    
df_stats = pd.DataFrame(stats)

df_stats['vol_tier'] = pd.qcut(df_stats['vol'], 3, labels=['low_vol', 'mid_vol', 'high_vol'])

df_stats['liq_tier'] = pd.qcut(df_stats['dollar_vol'], 2, labels=['mid_liq', 'high_liq'])

stratified_map = {}

for (sector, vol, liq), group in df_stats.groupby(['sector', 'vol_tier', 'liq_tier']):
    bucket_name = f"{sector}_{vol}_{liq}".replace(" ", "_")
    stratified_map[bucket_name] = group['path'].tolist()

with open('stratified_metadata.json', 'w') as j:
    json.dump(stratified_map, j, indent=4)

print(f"Created {len(stratified_map)} distinct training buckets.")

Created 18 distinct training buckets.


**Now, we have stratified training sets for each distinct market sector, so we can train our models separately on all of them**